In [ ]:

from forget.model import Llama2Wrapper, Llama3Wrapper
from forget.chat import Chat
import torch as t
import os
import pandas as pd
import re
from tqdm.auto import tqdm

HF_TOKEN = os.getenv("HF_TOKEN")


In [ ]:
# llm = Llama2Wrapper(hf_token=HF_TOKEN, size="7b", use_chat=True, gpu_id=0)
llm = Llama3Wrapper(hf_token=HF_TOKEN, size="8b", use_chat=True, gpu_id=0)

In [ ]:
# chat = Chat(system_prompt="You are a helpful assistant.")
# chat.add_user_message("What is the capital of France?")
# response = llm.generate_from_chat(chat, max_new_tokens=100)
# print(response)

In [ ]:
df_good = pd.read_csv("store/good.csv")
df_bad = pd.read_csv("store/bad.csv")

# run baseline benchmark

In [ ]:
MCQ_SYSTEM = "You are answering multiple choice questions. Reply with ONLY the letter of the correct answer (A, B, C, or D). Do not explain."

def format_mcq_prompt(row):
    return (
        f"{row['question']}\n"
        f"A) {row['A']}\n"
        f"B) {row['B']}\n"
        f"C) {row['C']}\n"
        f"D) {row['D']}"
    )

def parse_answer(response: str) -> str:
    match = re.search(r'\b([ABCD])\b', response.split("assistant")[-1])
    return match.group(1) if match else ""

def evaluate_row(llm, row):
    chat = Chat(system_prompt=MCQ_SYSTEM)
    chat.add_user_message(format_mcq_prompt(row))
    raw = llm.generate_from_chat(chat, max_new_tokens=10, do_sample=False, temperature=1.0)
    parsed = parse_answer(raw)
    correct = int(parsed == row["answer"])
    return raw, parsed, correct


In [ ]:
results = []
pbar = tqdm(df_good.iterrows(), total=len(df_good))
for i, row in pbar:
    raw, parsed, correct = evaluate_row(llm, row)
    results.append({"model_output": raw, "parsed": parsed, "correct": correct})
    acc = sum(r["correct"] for r in results) / len(results)
    pbar.set_description(f"acc: {acc:.1%}")

res_df = pd.concat([df_good, pd.DataFrame(results)], axis=1)
res_df.to_csv("store/llama3_baseline.csv", index=False)

# calculate steering vector

# run steered benchmark